# Laboratorio 8: Costruire il metro

---

# Lab 8: Building the yardstick

**Come si usa.** Esegui le celle dall'alto in basso (Maiusc+Invio). Serve un account Google gratuito
per usare Colab. Il testo completo del laboratorio e' sul sito: https://corso-rag.giuseppebosi.com/moduli/08-valutare-il-sistema/LAB/

---

**How to use it.** Run the cells from top to bottom (Shift+Enter). Colab needs a free Google account.
The full text of the lab is on the site: https://corso-rag.giuseppebosi.com/en/moduli/08-valutare-il-sistema/LAB/

In [ ]:
# Prepara l'ambiente / Set up the environment (about 1 minute)
import os, sys, subprocess
if sys.version_info[:2] not in [(3, 11), (3, 12)]:
    print("ATTENZIONE: questo corso e' provato con Python 3.11 o 3.12; hai "
          f"{sys.version_info.major}.{sys.version_info.minor}. "
          "WARNING: the course is tested on Python 3.11 or 3.12.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(BASE, "rag-chatbot-course")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/JoeBosi/rag-chatbot-course.git", REPO_DIR], check=True)
os.chdir(os.path.join(REPO_DIR, "corso"))
!pip install -q -r requirements.txt
print("Cartella di lavoro / working folder:", os.getcwd())

## La chiave gratuita / The free key

Questo laboratorio chiama un modello: serve una chiave **Groq**, gratuita e senza carta di credito
([CHIAVI.md](https://corso-rag.giuseppebosi.com/CHIAVI/) spiega come ottenerla in due minuti).
Poi in Colab: icona della **chiave** nella barra a sinistra, **Aggiungi nuovo segreto**, nome
`GROQ_API_KEY`, valore la tua chiave, e attiva **Accesso da notebook**. Se non la imposti, la cella
te la chiede al momento. La chiave non finisce mai nel notebook.

---

This lab calls a model, so you need a free **Groq** key, no credit card
([CHIAVI.md](https://corso-rag.giuseppebosi.com/en/CHIAVI/) shows how to get one in two minutes).
Then in Colab: the **key** icon in the left bar, **Add new secret**, name `GROQ_API_KEY`, value your
key, and switch on **Notebook access**. If you skip this, the cell asks for the key when you run it.
The key is never saved in the notebook.

In [ ]:
# La chiave Groq / The Groq key. Mai scritta nel notebook / never written in the notebook.
import os
from getpass import getpass

def leggi_chiave(nome="GROQ_API_KEY"):
    if os.environ.get(nome):
        return os.environ[nome]
    try:
        from google.colab import userdata          # Colab: icona della chiave, a sinistra
        return userdata.get(nome)
    except Exception:
        pass
    try:
        return getpass(f"Incolla {nome} / paste {nome}: ")
    except Exception:
        return ""

chiave = leggi_chiave()
if chiave:
    os.environ["GROQ_API_KEY"] = chiave
    os.environ["LLM_PROVIDER"] = "groq"
    print("Chiave caricata (non viene mostrata) / key loaded (not displayed).")
else:
    print("Nessuna chiave. I passaggi che chiamano il modello si fermeranno con "
          "il messaggio 'Manca GROQ_API_KEY'. Vedi la cella sopra.\n"
          "No key. Steps that call the model will stop with 'Manca GROQ_API_KEY'. See the cell above.")
![ -f .env ] || cp .env.example .env

Le celle con un'etichetta **[serve la chiave]** chiamano il modello. Senza chiave si fermano con
`Manca GROQ_API_KEY in .env`: non e' un guasto, inserisci la chiave come spiegato sopra e rilancia.
Le altre celle funzionano anche senza.

---

Cells labelled **[needs the key]** call the model. Without a key they stop with
`Manca GROQ_API_KEY in .env` (the message is Italian): this is not a fault, add the key as explained
above and run the cell again. The other cells work without it.

Serve l'archivio **completo** (12 documenti). Il giro di 12 domande con la pausa di 30 secondi dura sei-sette minuti. Il gateway del modulo 2 gira come processo di sfondo (celle qui sotto).

---

It needs the **complete** archive (12 documents). The round of 12 questions with the 30 s pause takes six or seven minutes. The module 2 gateway runs as a background process (cells below).

## Il gateway

Come nel laboratorio 2, il gateway gira come processo di sfondo, in un ambiente virtuale suo, sul solo indirizzo `127.0.0.1`. L'installazione richiede un paio di minuti.

---

## The gateway

As in lab 2, the gateway runs as a background process in its own virtual environment, on `127.0.0.1` only. Installing takes a couple of minutes.

In [ ]:
!python -m venv .venv-gateway
!.venv-gateway/bin/pip install -q -r requirements-gateway.txt
!mkdir -p memoria/gateway
!cp moduli/02-chiavi-gateway-e-quote/starter/config.yaml memoria/gateway/

In [ ]:
import subprocess, time, os
GATEWAY = subprocess.Popen(
    [".venv-gateway/bin/litellm", "--config", "memoria/gateway/config.yaml",
     "--host", "127.0.0.1", "--port", "4000"],
    stdout=open("gateway.log", "w"), stderr=subprocess.STDOUT, env=dict(os.environ))
for _ in range(180):
    time.sleep(2)
    if "Uvicorn running on http://127.0.0.1:4000" in open("gateway.log").read():
        print("Gateway acceso su http://127.0.0.1:4000 / gateway is up")
        break
else:
    print(open("gateway.log").read()[-1500:])

In [ ]:
# Come nel laboratorio 5: cognee parla col gateway locale / as in lab 5: cognee talks to the local gateway
os.environ.update(
    LLM_PROVIDER="custom", LLM_ENDPOINT="http://localhost:4000/v1",
    LLM_MODEL="openai/memoria-veloce", LLM_API_KEY="sk-memoria-locale")

In [ ]:
!mkdir -p dati
if not os.path.isdir("dati/cognee"):
    !python memoria/quaderno.py aggiungi
    !python memoria/quaderno.py digerisci

## Starter o soluzione? / Starter or solution?

Nel laboratorio scrivi tu alcune funzioni (i TODO). Qui sotto scegli:
`USA_SOLUZIONE = True` copia la soluzione, cosi' tutte le celle girano subito;
`USA_SOLUZIONE = False` copia lo **starter** con i TODO: apri il file nel pannello **File** a sinistra
(cartella `rag-chatbot-course/corso/memoria`), scrivi la funzione, salva e rilancia le celle.

---

In the lab you write some functions yourself (the TODOs). Choose below:
`USA_SOLUZIONE = True` copies the solution so every cell runs straight away;
`USA_SOLUZIONE = False` copies the **starter** with the TODOs: open the file in the **Files** panel on
the left (folder `rag-chatbot-course/corso/memoria`), write the function, save, and run the cells again.

In [ ]:
USA_SOLUZIONE = True   # False = parti dallo starter e scrivi tu i TODO
MOD = "moduli/08-valutare-il-sistema"
ORIGINE = "soluzione" if USA_SOLUZIONE else "starter"
!mkdir -p memoria
!cp $MOD/$ORIGINE/valuta.py memoria/

## Esercizio 1: l'insieme di valutazione

Dodici domande. Prima di lanciare, scrivi la tua previsione: passa o non passa?

---

## Exercise 1: the evaluation set

Twelve questions. Before running, write your prediction: pass or fail?

In [ ]:
!cat progetto-finale/domande.yaml

## Esercizi 2-5: il giro completo **[serve la chiave]**

Alla fine un riassunto per tipo di domanda e un file in `dati/valutazioni/`.

---

## Exercises 2-5: the full round **[needs the key]**

At the end a summary by question type and a file in `dati/valutazioni/`.

In [ ]:
!python memoria/valuta.py

Il manuale del rasaerba e' un PDF scansionato: senza OCR non contiene caratteri leggibili. Deve stampare `0`.

---

The lawn-mower manual is a scanned PDF: without OCR it holds no readable characters. It must print `0`.

In [ ]:
import pypdf
print(len(pypdf.PdfReader("progetto-finale/corpus/manuali/greencut-rm46.pdf").pages[0].extract_text()))

## Esercizio 6: misurare un cambiamento **[serve la chiave]**

Solo recupero, senza grafo: confronta i due riassunti per tipo.

---

## Exercise 6: measure a change **[needs the key]**

Retrieval only, no graph: compare the two summaries by type.

In [ ]:
!python memoria/valuta.py --tipo testo

## Verifica

Non interroga il modello: giudica risposte finte.

---

## Check

It does not call the model: it grades fake answers.

In [ ]:
!python moduli/08-valutare-il-sistema/verifica.py